In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------


# Two small washes beat one big one (Illustration 11.2-9)

This notebook works **Illustration 11.2-9**: the same acetone-water mixture as
Illustration 11.2-8, but contacted with methyl isobutyl ketone (MIK) in **two stages of
1 kg each** instead of one contact with 3 kg.

**The claim being tested.** Less solvent, better recovery. That sounds like something
for nothing, and it is not — it is the same trade every countercurrent extractor in a
plant is built on, and the notebook puts a number on both halves of it.

**What is new relative to Illustration 11.2-8.** Nothing thermodynamic. Every stage is
the same three steps — combine the streams, find the tie line through the feed point,
split by the lever rule — and the only new idea is that *the water-rich product of one
stage is the feed to the next*. That is what makes it a process rather than an
experiment.

**Why two stages is where the interesting arithmetic is.** With one stage the solvent
sees the feed at its richest and leaves at equilibrium with the exhausted raffinate. With
two, each fresh kilogram of solvent meets a stream that still has acetone in it. The
driving force is used twice, and the notebook measures what that buys.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst  
August 2026


In [2]:
import sys; sys.path.append("..")
import numpy as np
from scipy.interpolate import PchipInterpolator

from thermo import ternary
from thermo.lle import mix_streams, tie_line_split

# The same coexistence data as Figure 11.2-8: D. F. Othmer, R. E. White and
# E. Trueger, Ind. Eng. Chem. 33, 1240 (1941), the reference in SIS footnote 13,
# reprinted in the chapter as the table below that figure. Repeated here rather than
# imported so this notebook stands on its own in Colab.
OTHMER_1941 = np.array([
    [93.2,   4.60,   2.33], [27.4,  48.4,  24.1],
    [77.3,  18.95,   3.86], [20.1,  46.3,  33.5],
    [71.0,  24.4,    4.66], [ 2.12,  3.73, 94.2],
    [65.5,  28.9,    5.53], [ 3.23, 20.9,  75.8],
    [54.7,  37.6,    7.82], [ 5.01, 30.9,  64.2],
    [46.2,  43.2,   10.7 ], [12.4,  42.7,  45.0],
    [38.3,  47.0,   14.8 ], [20.5,  46.6,  32.8],
    [32.8,  48.3,   18.8 ], [25.9,  50.7,  23.4],
])
MIK, ACE, WAT = 0, 1, 2

binodal = OTHMER_1941 / OTHMER_1941.sum(axis=1)[:, None]
binodal = binodal[np.argsort(-binodal[:, MIK])][:, [ACE, MIK, WAT]]
_xy = ternary.to_xy(binodal)
_s = np.concatenate([[0], np.cumsum(np.hypot(*(_xy[1:] - _xy[:-1]).T))])
CURVE = np.column_stack([PchipInterpolator(_s, binodal[:, k])(
    np.linspace(0, _s[-1], 400)) for k in range(3)])
CURVE /= CURVE.sum(axis=1)[:, None]

def on_curve(comp):
    """Distance, in weight fraction, from a composition to the measured binodal."""
    p = ternary.to_xy(np.asarray(comp)[[ACE, MIK, WAT]]).ravel()
    return np.hypot(*(ternary.to_xy(CURVE) - p).T).min()

print(f"  binodal loaded: {len(OTHMER_1941)} measured compositions at 298.15 K")

  binodal loaded: 16 measured compositions at 298.15 K


## One stage, written once

Both stages are the same calculation, so it is written once. Given a solvent
stream and a feed stream and the tie line SIS reads off Figure 11.2-8, a stage returns
the two product streams.

**The tie lines are inputs here, not results.** They were read off the figure with a
straightedge, because the chapter reprints Othmer's coexistence curve but not his
tie-line table. What the notebook *can* do — and does, below — is check that both ends
of each read-off tie line actually land on the measured curve.

In [3]:
def stage(solvent_kg, solvent, feed_kg, feed, mik_rich, water_rich, name):
    """One contacting stage. Returns (amount, composition) of the water-rich product."""
    total, z = mix_streams([solvent_kg, feed_kg], [solvent, feed])
    LI, LII = tie_line_split(z, mik_rich, water_rich, total=total)

    print(f"  {name}")
    print(f"    combined feed  {total:.3f} kg at "
          f"{100*z[MIK]:.1f} / {100*z[ACE]:.1f} / {100*z[WAT]:.1f} wt % MIK/acetone/water")
    print(f"    tie line ends sit {100*on_curve(mik_rich):.2f} and "
          f"{100*on_curve(water_rich):.2f} wt % from the measured curve")
    print(f"    MIK-rich   {LI:.3f} kg")
    print(f"    water-rich {LII:.3f} kg")
    return LII, water_rich, LI, mik_rich

AQUEOUS = np.array([0.0, 0.6, 0.4])      # 60 wt % acetone, 40 wt % water
PURE_MIK = np.array([1.0, 0.0, 0.0])

# Tie lines as SIS reads them off Fig. 11.2-8.
TIE_1 = (np.array([62.0, 32.0,  6.0]) / 100, np.array([2.0, 23.0, 75.0]) / 100)
TIE_2 = (np.array([88.4,  7.6,  4.0]) / 100, np.array([2.0,  3.0, 95.0]) / 100)

In [4]:
L2_kg, L2_comp, E1_kg, E1_comp = stage(1.0, PURE_MIK, 1.0, AQUEOUS, *TIE_1,
                                       name="Stage 1: 1 kg MIK + 1 kg of the feed")
print("    SIS: L_I = 1.59 kg, L_II = 0.41 kg\n")

R2_kg, R2_comp, E2_kg, E2_comp = stage(1.0, PURE_MIK, L2_kg, L2_comp, *TIE_2,
                                       name="Stage 2: 1 kg fresh MIK + stage 1's water-rich product")
print("    SIS: L_I = 1.134 kg, L_II = 0.276 kg")

  Stage 1: 1 kg MIK + 1 kg of the feed
    combined feed  2.000 kg at 50.0 / 30.0 / 20.0 wt % MIK/acetone/water
    tie line ends sit 0.22 and 1.19 wt % from the measured curve
    MIK-rich   1.596 kg
    water-rich 0.404 kg
    SIS: L_I = 1.59 kg, L_II = 0.41 kg

  Stage 2: 1 kg fresh MIK + stage 1's water-rich product
    combined feed  1.404 kg at 71.8 / 6.6 / 21.6 wt % MIK/acetone/water
    tie line ends sit 1.17 and 0.79 wt % from the measured curve
    MIK-rich   1.133 kg
    water-rich 0.270 kg
    SIS: L_I = 1.134 kg, L_II = 0.276 kg


Both stages reproduce, and the small differences have two distinct causes.

**Stage 1** gives 1.596 / 0.404 kg against the printed 1.59 / 0.41. The chapter balances
on water alone; `tie_line_split` uses all three species by least squares, and with a tie
line read off a diagram the three balances disagree at about the third decimal.

**Stage 2** gives 1.133 / 0.270 against 1.134 / 0.276, and the reason is different: this
notebook feeds stage 2 the **unrounded 0.404 kg** that stage 1 actually produced, where
the chapter carries the rounded 0.41 kg forward. That is why the combined feed here is
71.8 / 6.6 / 21.6 wt % against the printed 71.5 / 6.7 / 21.8. Neither is wrong; the
printed route rounds once per stage, and rounding once per stage is what a hand
calculation does.

**What is genuinely new here is the third line of each block.** Every one of the four
read-off tie-line ends lands between **0.22 and 1.19 wt %** of Othmer's measured
coexistence curve. Those compositions have been carried through three editions of this
book without anything to check them against, because the measurements they came from
were never reprinted alongside them. They hold up.

In [5]:
kg = R2_kg * R2_comp
print("  the water-rich stream leaving stage 2")
print(f"    water    {kg[WAT]:.4f} kg     SIS 0.263")
print(f"    acetone  {kg[ACE]:.4f} kg     SIS 0.0083")
print(f"    MIK      {kg[MIK]:.4f} kg     SIS 0.0055")

fed_acetone = 0.6
print(f"\n  acetone left behind: {100*kg[ACE]/fed_acetone:.1f} % of what was fed")
print(f"  solvent used:        2.000 kg, of which {kg[MIK]:.4f} kg is lost in the water")

  the water-rich stream leaving stage 2
    water    0.2568 kg     SIS 0.263
    acetone  0.0081 kg     SIS 0.0083
    MIK      0.0054 kg     SIS 0.0055

  acetone left behind: 1.4 % of what was fed
  solvent used:        2.000 kg, of which 0.0054 kg is lost in the water


## Against the single stage

The comparison is the point of the illustration, so both cases are computed and put
side by side rather than described.

In [6]:
# Illustration 11.2-8, recomputed here so the comparison is not quoted from the page.
one_total, one_z = mix_streams([3.0, 1.0], [PURE_MIK, AQUEOUS])
ONE_TIE = (np.array([80.5, 15.5, 4.0]) / 100, np.array([2.0, 8.0, 90.0]) / 100)
one_LI, one_LII = tie_line_split(one_z, *ONE_TIE, total=one_total)
one_left = one_LII * ONE_TIE[1][ACE]

rows = (("one stage, 3 kg MIK", 3.0, one_left, one_LII * ONE_TIE[1][MIK]),
        ("two stages, 1 kg each", 2.0, kg[ACE], kg[MIK]))
print(f"  {'':24s} {'MIK used':>9s} {'acetone left':>13s} {'MIK lost':>9s}")
for name, used, left, lost in rows:
    print(f"  {name:24s} {used:8.1f} kg {left:11.4f} kg {lost:8.4f} kg")

print(f"\n  Two stages use {100*(1 - 2.0/3.0):.0f} % less solvent and leave "
      f"{one_left/kg[ACE]:.1f} times less acetone in the water.")
print(f"  Recovery goes from {100*(1 - one_left/fed_acetone):.1f} % to "
      f"{100*(1 - kg[ACE]/fed_acetone):.1f} %.")

                            MIK used  acetone left  MIK lost
  one stage, 3 kg MIK           3.0 kg      0.0224 kg   0.0056 kg
  two stages, 1 kg each         2.0 kg      0.0081 kg   0.0054 kg

  Two stages use 33 % less solvent and leave 2.8 times less acetone in the water.
  Recovery goes from 96.3 % to 98.6 %.


## Why staging wins, in one sentence

Equilibrium sets a *ratio*, not an amount. A single contact with 3 kg of solvent brings
the raffinate to equilibrium with a solvent phase that is already carrying acetone —
and the acetone it is carrying is what stops more from leaving. Splitting the same
solvent into two portions gives the second one a clean start, so the water is brought to
equilibrium twice against fresh solvent instead of once against loaded solvent.

**And the cost is not zero.** Two stages means two vessels, two settlers, two sets of
pumps and controls. The chapter says this plainly, and the sentence should survive the revision: *"this would
involve greater costs since more equipment is needed."* The economic optimum is not the
thermodynamic one, and finding it is a design course, not this chapter.

## Your turn

1. Three stages of 2/3 kg each use the same 2 kg of solvent. Work them, reading each tie
   line off the curve. Does the improvement over two stages match the improvement two
   stages gave over one, or is it smaller? Plot acetone remaining against stage count.
2. The stage function takes the tie line as an argument, which is the weak point. Replace
   it with a rule — assume the tie lines all pass through a common point outside the
   triangle, fit that point to the two the chapter gives, and re-solve. How much do the
   answers move?
3. Run stage 2 with the *MIK-rich* product recycled instead of discarded. That is a
   countercurrent cascade, and it is what a real plant does.
4. At what number of stages does the acetone left in the water stop falling appreciably?
   Argue from the diagram where that limit comes from — look at where the tie lines go as
   the raffinate approaches the water corner.